# Shoplifting detection from CCTV video

Re-run of the thesis experiments in a single notebook, with the bugs in the original code fixed.

1. Download the 50 Shoplifting and 50 normal videos from UCF-Crime.
2. Extract ResNet-50 V2 (ImageNet) features for every frame, once.
3. Train frame features → LSTM → P(shoplifting).
4. Evaluate with stratified 5-fold cross-validation, repeated over several seeds.
5. Compare against baselines, a shuffled-frames ablation and a pre-theft test.
6. Re-create the original thesis pipeline, bugs included, for comparison.

**Running.** On Colab, pick a T4 GPU runtime and run all. Features and results are saved to `MyDrive/shoplifting-rerun`, so running again after a disconnect resumes where it stopped. `SMOKE_TEST = True` does a quick check run on the 6 shortest videos per class (the numbers are meaningless). The full run is about 30–40 min of feature extraction plus about an hour of experiments on a T4.

**Changes from the original code**

| Original | Problem | Now | Section |
|---|---|---|---|
| 12 copy-pasted notebooks | two of them used the wrong frames or the wrong checkpoint | one notebook, one settings cell | 0 |
| `os.listdir()` for frame files | frames reached the LSTM in random order | frames kept in time order | 5 |
| `Masking` before the ResNet layer (Model A) | mask dropped, LSTM read the padded black frames | masking on the features, tested | 5, 6 |
| ResNet run every epoch (Model B) | 26–35 min per epoch | features computed once | 4 |
| `cv2.imread` (BGR) into an RGB model | colour channels swapped | converted to RGB | 4 |
| `int(fps) // target_fps` | wrong at 29.97 fps, fails below the target fps | sampling by timestamp | 4, 5 |
| LSTM(512), 20 epochs | best epoch was always 1 in Model B | smaller LSTM, dropout, early stopping | 6 |
| one 18-video test set | one video = 5.6 points | 5-fold CV × seeds | 7 |
| ROC from rounded 0/1 predictions | not a real ROC curve | AUC from probabilities | 7 |
| no baselines | no way to tell what the LSTM adds | 3 baselines + shuffled frames | 8 |
| clips include the theft | "pre-crime" never tested | test on footage before the theft | 10 |

## 0. Settings

In [ ]:
SMOKE_TEST = False        # quick check run on the 6 shortest videos per class

N_PER_CLASS = 50          # videos per class (UCF-Crime has 50 shoplifting videos)
BASE_FPS = 30             # feature extraction rate; lower rates are subsampled from it
FPS_SWEEP = [5, 15, 30]   # frame rates to compare, as in the thesis
MAIN_FPS = 15             # frame rate for the shuffled-frames run
MAX_FRAMES = 1000         # max sequence length (the thesis "_1" notebooks used 1000)
N_FOLDS = 5               # cross-validation folds
N_SEEDS = 3               # cross-validation repeats with different splits
MIN_PRE_THEFT_SEC = 10    # pre-theft test: minimum footage before the theft
PRE_THEFT_MARGIN_SEC = 1  # pre-theft test: cut this many seconds before the theft starts

if SMOKE_TEST:
    N_PER_CLASS, BASE_FPS, FPS_SWEEP, MAIN_FPS, MAX_FRAMES, N_SEEDS = 6, 5, [5], 5, 300, 1
    MIN_PRE_THEFT_SEC = 3

## 1. Setup

The Keras backend has to be set before Keras is imported. This picks whichever of JAX, TensorFlow or PyTorch can see a GPU.

On Colab, features and results go to Drive. Videos go to the local disk, since they aren't needed once the features exist.

In [ ]:
import gc, http.client, importlib.util, os, re, shutil, tempfile, time, urllib.request, zipfile, zlib


def pick_backend():
    """Prefer a backend with GPU access: JAX, then TensorFlow, then PyTorch."""
    if os.environ.get("KERAS_BACKEND"):
        return os.environ["KERAS_BACKEND"]
    if importlib.util.find_spec("jax"):
        import jax
        if any(d.platform == "gpu" for d in jax.devices()):
            return "jax"
    if importlib.util.find_spec("tensorflow"):
        import tensorflow as tf
        if tf.config.list_physical_devices("GPU"):
            return "tensorflow"
    if importlib.util.find_spec("torch"):
        import torch
        if torch.cuda.is_available():
            return "torch"
    return "jax"  # CPU fallback


os.environ["KERAS_BACKEND"] = pick_backend()

import cv2
import keras
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from keras import layers
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    drive.mount("/content/drive")
    WORK_DIR = "/content/drive/MyDrive/shoplifting-rerun"
    VIDEO_DIR = "/content/videos"
else:
    WORK_DIR = os.path.abspath("work")
    VIDEO_DIR = os.path.join(WORK_DIR, "videos")

FEATURE_DIR = os.path.join(WORK_DIR, f"features_{BASE_FPS}fps")
CACHE_DIR = os.path.join("/content" if IN_COLAB else WORK_DIR, f"feature_cache_{BASE_FPS}fps")  # local disk
RUN_DIR = os.path.join(WORK_DIR, "runs_smoke" if SMOKE_TEST else "runs")
for d in (WORK_DIR, VIDEO_DIR, FEATURE_DIR, CACHE_DIR, RUN_DIR):
    os.makedirs(d, exist_ok=True)


def ram():
    """System RAM in use."""
    try:
        import psutil
    except ImportError:
        return "RAM: (install psutil to see usage)"
    vm = psutil.virtual_memory()
    return f"RAM in use: {(vm.total - vm.available) / 1e9:.1f} of {vm.total / 1e9:.1f} GB"


def describe_devices():
    backend = keras.backend.backend()
    if backend == "jax":
        import jax
        return ", ".join(str(d) for d in jax.devices())
    if backend == "tensorflow":
        import tensorflow as tf
        return ", ".join(d.name for d in tf.config.list_physical_devices())
    import torch
    return torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"


print(f"Keras {keras.__version__} on {keras.backend.backend()} | devices: {describe_devices()}")
print("Saving features and results to", WORK_DIR)
if "gpu" not in describe_devices().lower() and "cuda" not in describe_devices().lower():
    print("WARNING: no GPU found. On Colab: Runtime > Change runtime type > T4 GPU, then run again.")

# plot style
BLUE, ORANGE, GREY = "#2a78d6", "#eb6834", "#a3a29c"
CLASS_COLOURS = {"normal": BLUE, "shoplifting": ORANGE}
plt.rcParams.update({
    "figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": "#52514e", "xtick.color": "#52514e",
    "ytick.color": "#52514e", "axes.grid": True, "grid.color": "#ebeae6", "grid.linewidth": 0.8,
    "axes.axisbelow": True, "axes.titlesize": 11, "axes.titleweight": "bold", "legend.frameon": False,
})

## 2. Data

[UCF-Crime](https://www.crcv.ucf.edu/research/real-world-anomaly-detection-in-surveillance-videos/), from a [Hugging Face mirror](https://huggingface.co/datasets/jinmang2/ucf_crime) of the official archives:

- Shoplifting: 50 videos in `Anomaly-Videos-Part-4.zip` (6.6 GB, which also holds Stealing and Vandalism).
- Normal: 50 videos in `Normal_Videos_for_Event_Recognition.zip` (1.1 GB).

`RemoteFile` reads the remote zip through HTTP Range requests, so only the zip's central directory and the needed videos are downloaded.

The thesis used 60 shoplifting videos, but UCF-Crime only has 50, so the original split can't be rebuilt. This is a new experiment, not an exact re-run.

In [ ]:
HF_BASE = "https://huggingface.co/datasets/jinmang2/ucf_crime/resolve/main/"
CLASSES = ["normal", "shoplifting"]  # label 0 and label 1
SOURCES = {  # class -> (zip file on the mirror, folder inside the zip)
    "normal": ("Normal_Videos_for_Event_Recognition.zip", "Normal_Videos_for_Event_Recognition/"),
    "shoplifting": ("Anomaly-Videos-Part-4.zip", "Anomaly-Videos-Part-4/Shoplifting/"),
}
ANNOTATIONS_URL = HF_BASE + "UCF_Crimes-Train-Test-Split/Temporal_Anomaly_Annotation_for_Testing_Videos.txt"


class RemoteFile:
    """Read-only file over HTTP; every read() is a Range request."""

    def __init__(self, url):
        self.url, self.pos = url, 0
        with self._get(0, 0) as r:  # total size comes back in Content-Range
            self.size = int(r.headers["Content-Range"].split("/")[-1])

    def _get(self, start, end):
        req = urllib.request.Request(self.url, headers={"Range": f"bytes={start}-{end}"})
        r = urllib.request.urlopen(req, timeout=120)
        if r.status != 206:
            r.close()
            raise IOError(f"server ignored the Range request (HTTP {r.status})")
        return r

    def seekable(self):
        return True

    def tell(self):
        return self.pos

    def seek(self, offset, whence=0):
        self.pos = {0: 0, 1: self.pos, 2: self.size}[whence] + offset
        return self.pos

    def read(self, n=-1):
        if n is None or n < 0:
            n = self.size - self.pos
        n = min(n, self.size - self.pos)
        if n <= 0:
            return b""
        for attempt in range(5):
            try:
                with self._get(self.pos, self.pos + n - 1) as r:
                    data = r.read()
                if len(data) == n:
                    self.pos += n
                    return data
            except (OSError, http.client.HTTPException) as e:
                print(f"  network problem ({e}), retrying")
            time.sleep(2 ** attempt)
        raise IOError("download failed after 5 attempts")


def download_member(archive, member, dest_dir):
    """Extract one file from the remote zip, unless it's already there."""
    out = os.path.join(dest_dir, os.path.basename(member.filename))
    if os.path.exists(out) and os.path.getsize(out) == member.file_size:
        return out
    with archive.open(member) as src, open(out + ".part", "wb") as dst:
        shutil.copyfileobj(src, dst, length=16 * 2**20)  # 16 MB per request
    os.replace(out + ".part", out)
    return out


video_rows = []
for label, cls in enumerate(CLASSES):
    zip_name, folder = SOURCES[cls]
    archive = zipfile.ZipFile(RemoteFile(HF_BASE + zip_name))
    members = [m for m in archive.infolist() if m.filename.startswith(folder) and m.filename.lower().endswith(".mp4")]
    key = (lambda m: m.file_size) if SMOKE_TEST else (lambda m: m.filename)  # smoke test: smallest files
    members = sorted(members, key=key)[:N_PER_CLASS]
    print(f"{cls}: {len(members)} videos, {sum(m.file_size for m in members) / 1e9:.2f} GB")
    for i, m in enumerate(members):
        path = download_member(archive, m, VIDEO_DIR)
        video_rows.append({"video": os.path.basename(path), "label": label, "cls": cls, "path": path})
        print(f"\r  downloaded {i + 1}/{len(members)}", end="")
    print()

### Theft annotations

The UCF-Crime temporal annotations give the start and end frame of the theft (up to two per video) for 21 of the Shoplifting videos. They're used for the pre-theft test in section 10.

In [ ]:
def load_theft_annotations(url):
    """{file name: [(start_frame, end_frame), ...]} for the annotated shoplifting videos."""
    text = urllib.request.urlopen(url, timeout=60).read().decode()
    theft = {}
    for line in text.splitlines():
        parts = line.split()  # e.g. "Shoplifting001_x264.mp4  Shoplifting  1550  2000  -1  -1"
        if len(parts) >= 6 and parts[1] == "Shoplifting":
            s1, e1, s2, e2 = (int(p) for p in parts[2:6])
            theft[parts[0]] = [(s, e) for s, e in ((s1, e1), (s2, e2)) if s >= 0]  # -1 = no second theft
    return theft


def probe(path):
    cap = cv2.VideoCapture(path)
    fps, n_frames = cap.get(cv2.CAP_PROP_FPS) or 30.0, int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    return fps, n_frames


THEFT = load_theft_annotations(ANNOTATIONS_URL)
VIDEOS = pd.DataFrame(video_rows)
VIDEOS[["fps", "n_frames"]] = [probe(p) for p in VIDEOS.path]
VIDEOS["duration_s"] = VIDEOS.n_frames / VIDEOS.fps
VIDEOS["theft_start_s"] = [THEFT[v][0][0] / fps if v in THEFT else np.nan for v, fps in zip(VIDEOS.video, VIDEOS.fps)]
print(f"{len(VIDEOS)} videos; {VIDEOS.theft_start_s.notna().sum()} shoplifting videos have theft timestamps")
VIDEOS.head()

## 3. Data overview

Two possible shortcuts: the classes differing in length, and the scene alone (shop or not) being enough to tell them apart.

In [ ]:
print(VIDEOS.groupby("cls").duration_s.describe()[["count", "min", "50%", "max"]].round(0))

fig, ax = plt.subplots(figsize=(8, 3))
bins = np.logspace(np.log10(VIDEOS.duration_s.min()), np.log10(VIDEOS.duration_s.max()), 20)
for cls in CLASSES:
    ax.hist(VIDEOS.duration_s[VIDEOS.cls == cls], bins=bins, histtype="step", linewidth=2,
            color=CLASS_COLOURS[cls], label=cls)
ax.set_xscale("log")
ticks = [t for t in (5, 10, 20, 30, 60, 120, 300, 600, 1200, 1800) if bins[0] <= t <= bins[-1]]
ax.set_xticks(ticks, [str(t) for t in ticks])
ax.xaxis.set_minor_formatter(plt.NullFormatter())
ax.yaxis.set_major_locator(plt.MaxNLocator(integer=True))
ax.set_xlabel("video length (seconds, log scale)")
ax.set_ylabel("videos")
ax.set_title("Video length by class")
ax.legend()
plt.show()

In [ ]:
def middle_frame(path):
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) // 2)
    ok, frame = cap.read()
    cap.release()
    return cv2.cvtColor(frame, cv2.COLOR_BGR2RGB) if ok else np.zeros((240, 320, 3), np.uint8)


fig, axes = plt.subplots(2, 4, figsize=(12, 5.5), layout="constrained")
for row, cls in enumerate(CLASSES):
    sample = VIDEOS[VIDEOS.cls == cls].sample(min(4, (VIDEOS.cls == cls).sum()), random_state=0)
    for ax, (_, v) in zip(axes[row], sample.iterrows()):
        ax.imshow(middle_frame(v.path))
        ax.set_title(f"{cls}: {v.video.replace('_x264.mp4', '')}", fontsize=8, fontweight="normal")
    for ax in axes[row]:
        ax.axis("off")
fig.suptitle("Middle frame of sample videos")
plt.show()

## 4. Feature extraction

ResNet-50 V2 with ImageNet weights, frozen, global average pooling: 2048 features per frame. Because the backbone is frozen, features are computed once and saved instead of being recomputed every epoch.

- Frames are converted from BGR (OpenCV) to RGB.
- Pixels are scaled with `resnet_v2.preprocess_input`.
- Frames are sampled by timestamp at `BASE_FPS`, so 29.97 fps and low-fps videos are handled correctly.
- 64 frames per batch.

The thesis used the TensorFlow Hub ResNet-50 V2; this is the `keras.applications` version (same architecture, separately trained weights).

In [ ]:
IMG_SIZE, BATCH, FEAT_DIM = 224, 64, 2048


def build_extractor():
    """ResNet-50 V2 without the classifier head, average-pooled to 2048 features."""
    return keras.applications.ResNet50V2(include_top=False, weights="imagenet", pooling="avg",
                                         input_shape=(IMG_SIZE, IMG_SIZE, 3))


def read_frames(path, every_sec):
    """Yield (timestamp, 224x224 RGB frame), one frame every `every_sec` seconds, in order."""
    cap = cv2.VideoCapture(path)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    i, kept = 0, 0
    while cap.grab():  # grab() skips decoding; retrieve() decodes only the frames we keep
        t = i / fps
        if t >= kept * every_sec - 1e-6:
            ok, frame = cap.retrieve()
            if ok:
                frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                yield t, cv2.resize(frame, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            kept += 1
        i += 1
    cap.release()


def features_of(extractor, frames):
    """Features for up to BATCH frames, padded to a fixed batch shape."""
    n = len(frames)
    x = np.zeros((BATCH, IMG_SIZE, IMG_SIZE, 3), "float32")
    x[:n] = np.stack(frames)
    x = keras.applications.resnet_v2.preprocess_input(x)
    return np.asarray(extractor.predict_on_batch(x))[:n].astype("float16")


def extract_video(extractor, path):
    times, feats, batch = [], [], []
    for t, frame in read_frames(path, 1.0 / BASE_FPS):
        times.append(t)
        batch.append(frame)
        if len(batch) == BATCH:
            feats.append(features_of(extractor, batch))
            batch = []
    if batch:
        feats.append(features_of(extractor, batch))
    return np.array(times, "float32"), (np.concatenate(feats) if feats else np.zeros((0, FEAT_DIM), "float16"))


def feature_path(video):
    return os.path.join(FEATURE_DIR, video.replace(".mp4", ".npz"))

Each video's features are saved as soon as they're done. The features of an all-black frame are saved too; section 5 uses them to reproduce the original padding.

In [ ]:
todo = VIDEOS[[not os.path.exists(feature_path(v)) for v in VIDEOS.video]]
black_path = os.path.join(FEATURE_DIR, "_black_frame.npy")
extractor = build_extractor() if len(todo) or not os.path.exists(black_path) else None

if not os.path.exists(black_path):
    np.save(black_path, features_of(extractor, [np.zeros((IMG_SIZE, IMG_SIZE, 3), "uint8")])[0])

expected = (todo.duration_s * np.minimum(todo.fps, BASE_FPS)).sum()  # approximate frames left
start, done = time.time(), 0
for i, v in enumerate(todo.itertuples()):
    times, feats = extract_video(extractor, v.path)
    if len(times) == 0:
        print(f"  {v.video}: no readable frames, skipping")
        continue
    tmp = feature_path(v.video) + ".tmp.npz"
    np.savez(tmp, times=times, feats=feats)
    os.replace(tmp, feature_path(v.video))
    done += len(times)
    rate = done / (time.time() - start)
    print(f"[{i + 1}/{len(todo)}] {v.video}: {len(times)} frames | {rate:.0f} frames/s | "
          f"about {max(expected - done, 0) / rate / 60:.0f} min left")
print("All features ready." if len(todo) else "All features were already saved; nothing to do.")

The features total a few GB, too much to hold in Colab's RAM alongside training. Each video is copied once from Drive to local disk as `.npy` and memory-mapped (`mmap_mode="r"`).

In [ ]:
def load_features(video):
    """Cache one video's features on local disk as .npy (first call only), then memory-map them."""
    base = os.path.join(CACHE_DIR, video.replace(".mp4", ""))
    if not os.path.exists(base + ".feats.npy"):
        with np.load(feature_path(video)) as f:
            np.save(base + ".times.npy", f["times"])
            np.save(base + ".tmp.npy", f["feats"])
        os.replace(base + ".tmp.npy", base + ".feats.npy")
    return np.load(base + ".times.npy"), np.load(base + ".feats.npy", mmap_mode="r")


VIDEOS = VIDEOS[[os.path.exists(feature_path(v)) for v in VIDEOS.video]].reset_index(drop=True)
FEATS = {v: load_features(v) for v in VIDEOS.video}
BLACK = np.load(black_path).astype("float16")
VIDEOS["duration_s"] = [FEATS[v][0][-1] + 1 / BASE_FPS for v in VIDEOS.video]  # more exact than the file header
y = VIDEOS.label.values
print(f"{len(FEATS)} videos, {sum(len(t) for t, _ in FEATS.values()):,} frames, "
      f"{sum(f.nbytes for _, f in FEATS.values()) / 1e9:.1f} GB of features on disk (memory-mapped)")
print(ram())

## 5. Sequences

Frames are taken at the chosen fps. Videos with more than `MAX_FRAMES` frames are subsampled evenly; shorter ones are zero-padded and masked. Arrays are float16.

### Original bug 1: frame order

The original loader listed frame files with `os.listdir()` and never sorted them. `os.listdir()` returns file-system order, and `sorted()` would sort as text:

In [ ]:
with tempfile.TemporaryDirectory(dir=VIDEO_DIR) as d:
    for i in range(20):
        open(os.path.join(d, f"frame_{i}.jpg"), "w").close()
    print("os.listdir():", [int(n[6:-4]) for n in os.listdir(d)])
    print("sorted():    ", [int(n[6:-4]) for n in sorted(os.listdir(d))])
    print("by number:   ", sorted(int(n[6:-4]) for n in os.listdir(d)))

### Original bug 2: unmasked padding (Model A)

`Masking` sat in front of the custom ResNet layer, which doesn't support masks, so the mask was dropped. The black padding frames became non-zero features, and the LSTM's final state came after the padding.

`thesis_sequences()` reproduces the original loader: scrambled order, step subsampling, and black-frame padding without a mask. `sequences()` is the fixed version.

In [ ]:
def pick_indices(times, fps, max_frames):
    """Frames at `fps` (by timestamp), in time order; at most `max_frames`, spread evenly over the video."""
    slot = np.floor(times * fps + 1e-6).astype(int)  # 1/fps time slot of each frame
    idx = np.flatnonzero(np.r_[True, slot[1:] != slot[:-1]])  # first frame in each slot
    if len(idx) > max_frames:
        idx = idx[np.linspace(0, len(idx) - 1, max_frames).round().astype(int)]
    return idx


def frames_before(video, cutoffs):
    """A video's (times, feats), optionally cut at cutoffs[video] seconds."""
    times, feats = FEATS[video]
    if cutoffs is not None and video in cutoffs:
        keep = times < cutoffs[video]
        times, feats = times[keep], feats[keep]
    return times, feats


def sequences(fps, max_frames=MAX_FRAMES, shuffle=False, cutoffs=None):
    """Fixed pipeline: time order, even subsampling, zero padding (masked by the model)."""
    X = np.zeros((len(VIDEOS), max_frames, FEAT_DIM), "float16")
    for i, v in enumerate(VIDEOS.video):
        times, feats = frames_before(v, cutoffs)
        idx = pick_indices(times, fps, max_frames)
        if shuffle:  # shuffled-frames run
            idx = np.random.default_rng(zlib.crc32(v.encode())).permutation(idx)
        X[i, :len(idx)] = feats[idx]
    return X


def thesis_sequences(fps, max_frames=MAX_FRAMES):
    """The original data loader, bugs included."""
    X = np.empty((len(VIDEOS), max_frames, FEAT_DIM), "float16")
    X[:] = BLACK  # padding = black-frame features
    for i, v in enumerate(VIDEOS.video):
        times, feats = FEATS[v]
        files = pick_indices(times, fps, 10**9)  # all frames extracted at this fps
        files = files[np.random.default_rng(zlib.crc32(v.encode())).permutation(len(files))]  # os.listdir order
        step = max(1, len(files) // max_frames)  # original step subsampling
        chosen = files[::step][:max_frames]
        X[i, :len(chosen)] = feats[chosen]
    return X

Padding by frame rate. For long videos the `MAX_FRAMES` cap, not `fps`, sets the effective sampling rate.

In [ ]:
rows = []
for fps in FPS_SWEEP:
    n = np.array([len(pick_indices(FEATS[v][0], fps, 10**9)) for v in VIDEOS.video])
    used = np.minimum(n, MAX_FRAMES)
    rows.append({"fps": fps,
                 "median frames available": int(np.median(n)),
                 "videos cut down to MAX_FRAMES": f"{(n > MAX_FRAMES).mean():.0%}",
                 "average share of sequence that is padding": f"{(1 - used / MAX_FRAMES).mean():.0%}",
                 "median frames per second actually used": round(float(np.median(used / VIDEOS.duration_s)), 1)})
pd.DataFrame(rows).set_index("fps")

## 6. Models

### LSTM

Same structure as the thesis: Masking → Dense(128, relu) → LSTM → Dense(1, sigmoid). The LSTM is 128 units instead of 512, with dropout 0.3. The original model (about 1.3M weights, 84 training videos) reached 100% training accuracy by epoch 4 while validation loss went up.

`build_thesis_lstm()` is the original configuration.

In [ ]:
def build_lstm(units=128, proj=128, dropout=0.3, mask=True, lr=1e-3):
    inputs = keras.Input(shape=(None, FEAT_DIM))
    x = layers.Masking(mask_value=0.0)(inputs) if mask else inputs
    x = layers.Dense(proj, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    x = layers.LSTM(units)(x)
    x = layers.Dropout(dropout)(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs)
    model.compile(optimizer=keras.optimizers.Adam(lr), loss="binary_crossentropy", metrics=["accuracy"])
    return model


def build_thesis_lstm():
    """Original configuration: LSTM(512), no dropout, padding not masked."""
    return build_lstm(units=512, proj=128, dropout=0.0, mask=False)


build_lstm().summary()

### Masking check

With untrained weights, the masked model's output shouldn't change with the amount of padding. The thesis-style model (black-frame padding, no mask) should.

In [ ]:
keras.utils.set_random_seed(0)
fixed_model, thesis_model = build_lstm(), build_thesis_lstm()
times, feats = FEATS[VIDEOS.video[0]]
real = feats[pick_indices(times, FPS_SWEEP[0], 100)].astype("float32")[None]  # one video, up to 100 frames


def padded(x, n_pad, value):
    return np.concatenate([x, np.broadcast_to(value, (1, n_pad, FEAT_DIM))], axis=1)


for n_pad in [0, 100, 500]:
    p_fixed = float(fixed_model.predict(padded(real, n_pad, 0.0), verbose=0)[0, 0])
    p_thesis = float(thesis_model.predict(padded(real, n_pad, BLACK), verbose=0)[0, 0])
    print(f"{n_pad:4d} padding steps | fixed model: {p_fixed:.6f} | thesis-style model: {p_thesis:.6f}")

The thesis-style output changes once padding is added and then levels off: after enough identical black frames the LSTM state no longer depends on the video. With mostly-padding sequences (5 fps, `MAX_FRAMES = 1000`) that would explain Model A sitting at chance at 5 fps.

### Baselines

Logistic regression on:

- log video length;
- the middle frame's features;
- the mean of all frame features (same content as the LSTM input, no order).

`C=0.1` for the 2048-feature inputs.

In [ ]:
def summary_features(kind, cutoffs=None):
    """One row per video: its length, its middle frame, or the average of its frames."""
    rows = []
    for v in VIDEOS.video:
        times, feats = frames_before(v, cutoffs)
        if kind == "length":
            rows.append([np.log(times[-1] + 1)])
        elif kind == "one_frame":
            rows.append(feats[len(feats) // 2])
        else:
            rows.append(feats.mean(axis=0, dtype="float32"))
    return np.array(rows, "float32")


def logistic_trainer(C):
    def fit_and_predict(X, train, seed):
        model = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=5000))
        model.fit(X[train], y[train])
        return (lambda X: model.predict_proba(X)[:, 1]), {}
    return fit_and_predict

## 7. Evaluation

With 18 test videos, one video is 5.6 percentage points. 95% Wilson intervals:

In [ ]:
def wilson_interval(k, n, z=1.96):
    """95% confidence interval for k correct out of n."""
    p = k / n
    centre = (p + z * z / (2 * n)) / (1 + z * z / n)
    half = z * np.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / (1 + z * z / n)
    return centre - half, centre + half


for k, n in [(14, 18), (17, 18), (78, 100)]:
    lo, hi = wilson_interval(k, n)
    print(f"{k:3d}/{n} correct = {k / n:.0%}, 95% CI {lo:.0%} to {hi:.0%}")

- Stratified 5-fold cross-validation, repeated with `N_SEEDS` different splits. Every video is tested once per seed.
- 15% of each training fold is held out for validation. The weights from the epoch with the lowest validation loss are kept, with early stopping after `patience` epochs. The test fold isn't used for any decision.
- Accuracy uses a 0.5 threshold. AUC is computed from the predicted probabilities.
- Each fold's predictions are saved as a CSV, so a disconnect only loses the fold in progress.

In [ ]:
class KeepBest(keras.callbacks.Callback):
    """Restore the weights from the epoch with the lowest validation loss; optionally stop early."""

    def __init__(self, patience=None):
        super().__init__()
        self.patience = patience

    def on_train_begin(self, logs=None):
        self.best_loss, self.best_epoch, self.best_weights = np.inf, -1, None

    def on_epoch_end(self, epoch, logs=None):
        if logs["val_loss"] < self.best_loss:
            self.best_loss, self.best_epoch, self.best_weights = logs["val_loss"], epoch, self.model.get_weights()
        elif self.patience is not None and epoch - self.best_epoch >= self.patience:
            self.model.stop_training = True

    def on_train_end(self, logs=None):
        if self.best_weights is not None:
            self.model.set_weights(self.best_weights)


def lstm_trainer(build, epochs, batch_size, patience):
    def fit_and_predict(X, train, seed):
        fit, val = train_test_split(train, test_size=0.15, stratify=y[train], random_state=seed)  # split indices, copy X once
        keras.backend.clear_session()  # free memory from previous models
        gc.collect()
        keras.utils.set_random_seed(seed)
        model = build()
        keep = KeepBest(patience)
        model.fit(X[fit], y[fit], validation_data=(X[val], y[val]), epochs=epochs,
                  batch_size=batch_size, callbacks=[keep], verbose=0)
        predict = lambda X: model.predict(X, batch_size=16, verbose=0).ravel()
        return predict, {"best_epoch": keep.best_epoch + 1}
    return fit_and_predict


def run_cv(name, inputs, fit_and_predict):
    """Cross-validate one experiment. `inputs` maps a view name ("full", "pre_theft") to an array with
    one row per video; the model trains on "full" and is tested on every view."""
    results = []
    for seed in range(N_SEEDS):
        folds = StratifiedKFold(N_FOLDS, shuffle=True, random_state=seed).split(np.zeros(len(y)), y)
        for fold, (train, test) in enumerate(folds):
            path = os.path.join(RUN_DIR, f"{re.sub(r'[^A-Za-z0-9]+', '_', name)}__seed{seed}__fold{fold}.csv")
            if os.path.exists(path):  # done in an earlier session
                results.append(pd.read_csv(path))
                continue
            start = time.time()
            predict, info = fit_and_predict(inputs["full"], train, seed)
            df = pd.concat([pd.DataFrame({"experiment": name, "seed": seed, "fold": fold, "view": view,
                                          "video": VIDEOS.video.values[test], "label": y[test],
                                          "prob": predict(X[test]), **info})
                            for view, X in inputs.items()])
            df.to_csv(path, index=False)
            results.append(df)
            print(f"  {name} | seed {seed} fold {fold}: {time.time() - start:.0f}s"
                  + (f", best epoch {info['best_epoch']}" if "best_epoch" in info else "") + f" | {ram()}")
    return pd.concat(results, ignore_index=True)


def accuracy(d):
    return float(((d.prob > 0.5) == d.label).mean())


def auc(d):
    return roc_auc_score(d.label, d.prob) if d.label.nunique() == 2 else np.nan


def summarize(results, view="full"):
    """One row per experiment. Accuracy and AUC are computed over all videos for each seed, then averaged."""
    d = results[results.view == view]
    if view == "pre_theft":
        d = d[d.video.isin(list(PRE_CUT))]
    rows = []
    for name, g in d.groupby("experiment", sort=False):
        seeds = [s for _, s in g.groupby("seed")]
        folds = [accuracy(f) for _, f in g.groupby(["seed", "fold"])]
        rows.append({"experiment": name,
                     "accuracy": np.mean([accuracy(s) for s in seeds]),
                     "AUC": np.nanmean([auc(s) for s in seeds]),
                     "worst fold": min(folds), "best fold": max(folds),
                     "median best epoch": g["best_epoch"].dropna().median() if "best_epoch" in g else np.nan})
    return pd.DataFrame(rows).set_index("experiment").round(2)

## 8. Experiments

Baselines take seconds; each LSTM experiment is `N_FOLDS × N_SEEDS` trainings of under a minute each on a T4. Progress lines show RAM use.

### 8.1 Baselines

The pre-theft cut-offs (section 10) are set up here so every model is also tested on pre-theft clips.

In [ ]:
def pre_theft_cutoffs(seed=0):
    """Seconds of footage to keep per video for the pre-theft test.
    Shoplifting videos: everything up to the theft (minus a margin). Normal videos: a length drawn from the
    shoplifting cut-offs, so both classes end up with the same distribution of lengths."""
    rng = np.random.default_rng(seed)
    pos = VIDEOS[(VIDEOS.label == 1) & (VIDEOS.theft_start_s >= MIN_PRE_THEFT_SEC)]
    cut = {v: s - PRE_THEFT_MARGIN_SEC for v, s in zip(pos.video, pos.theft_start_s)}
    lengths = np.array(list(cut.values()))
    if len(lengths):
        for v in VIDEOS.video[VIDEOS.label == 0]:
            cut[v] = float(rng.choice(lengths))
    return cut


PRE_CUT = pre_theft_cutoffs()
n_pos = int((VIDEOS.label[VIDEOS.video.isin(list(PRE_CUT))] == 1).sum())
print(f"Pre-theft test: {n_pos} shoplifting videos with at least {MIN_PRE_THEFT_SEC}s before the theft, "
      f"and {len(PRE_CUT) - n_pos} normal videos cut to matching lengths")


def with_pre_theft(build_inputs):
    """{"full": inputs from whole videos, "pre_theft": the same from footage before the theft}."""
    views = {"full": build_inputs(None)}
    if n_pos:
        views["pre_theft"] = build_inputs(PRE_CUT)
    return views


RESULTS = []
for kind, name in [("length", "baseline: video length only"),
                   ("one_frame", "baseline: one frame"),
                   ("mean", "baseline: average of all frames")]:
    RESULTS.append(run_cv(name, with_pre_theft(lambda cut: summary_features(kind, cut)),
                          logistic_trainer(C=1.0 if kind == "length" else 0.1)))
summarize(pd.concat(RESULTS))

### 8.2 Original thesis pipeline

Scrambled frame order, unmasked black-frame padding, LSTM(512) without dropout, 20 epochs, best epoch kept. Run at the lowest and highest frame rate. Batch size 8 instead of 1, for speed.

In [ ]:
THESIS_TRAINING = dict(epochs=20, batch_size=8, patience=None)
for fps in sorted({FPS_SWEEP[0], FPS_SWEEP[-1]}):
    X = thesis_sequences(fps)
    RESULTS.append(run_cv(f"thesis pipeline @ {fps} fps", {"full": X},
                          lstm_trainer(build_thesis_lstm, **THESIS_TRAINING)))
    del X
    gc.collect()
summarize(pd.concat(RESULTS))

### 8.3 Fixed LSTM by frame rate

In [ ]:
FIXED_TRAINING = dict(epochs=40, batch_size=8, patience=8)
for fps in FPS_SWEEP:
    views = with_pre_theft(lambda cut: sequences(fps, cutoffs=cut))
    RESULTS.append(run_cv(f"fixed LSTM @ {fps} fps", views, lstm_trainer(build_lstm, **FIXED_TRAINING)))
    del views
    gc.collect()
summarize(pd.concat(RESULTS))

### 8.4 Shuffled frames

The same LSTM with each video's frames shuffled, in training and testing. If accuracy holds up, the model isn't using temporal order.

In [ ]:
X = sequences(MAIN_FPS, shuffle=True)
RESULTS.append(run_cv(f"fixed LSTM, frames shuffled @ {MAIN_FPS} fps", {"full": X},
                      lstm_trainer(build_lstm, **FIXED_TRAINING)))
del X
gc.collect()
ALL = pd.concat(RESULTS, ignore_index=True)
ALL.to_csv(os.path.join(WORK_DIR, "all_predictions_smoke.csv" if SMOKE_TEST else "all_predictions.csv"), index=False)
summarize(ALL)

## 9. Results

Grey dots are single test folds (20 videos each). Blue dots are accuracy over all videos, averaged across seeds. The dashed line is chance.

In [ ]:
def plot_results(results, view="full"):
    d = results[results.view == view]
    if view == "pre_theft":
        d = d[d.video.isin(list(PRE_CUT))]
    names = list(dict.fromkeys(d.experiment))[::-1]  # first experiment at the top
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 0.5 * len(names) + 1.5), sharey=True)
    for i, name in enumerate(names):
        g = d[d.experiment == name]
        folds = [accuracy(f) for _, f in g.groupby(["seed", "fold"])]
        jitter = np.random.default_rng(i).uniform(-0.18, 0.18, len(folds))
        ax1.scatter(folds, i + jitter, s=16, color=GREY, linewidths=0, label="one test fold" if i == 0 else None)
        ax1.scatter([np.mean([accuracy(s) for _, s in g.groupby("seed")])], [i], s=70, color=BLUE,
                    edgecolors="white", linewidths=1.5, zorder=3, label="all videos" if i == 0 else None)
        ax2.scatter([np.nanmean([auc(s) for _, s in g.groupby("seed")])], [i], s=70, color=BLUE,
                    edgecolors="white", linewidths=1.5, zorder=3)
    for ax, title in [(ax1, "Accuracy"), (ax2, "AUC")]:
        ax.axvline(0.5, color="#52514e", linestyle="--", linewidth=1)
        ax.set_xlim(0, 1.02)
        ax.set_title(title, loc="left")
        ax.grid(axis="y", visible=False)
    ax1.set_yticks(range(len(names)), names)
    ax1.legend(loc="upper left", bbox_to_anchor=(0, -0.12), ncol=2, fontsize=8)
    plt.tight_layout()
    plt.show()


plot_results(ALL)
summarize(ALL)

Comparisons:

- Fixed LSTM vs. average of all frames and vs. shuffled frames: does frame order add anything?
- One-frame baseline: how much comes from scene appearance alone?
- Length baseline: is video length a shortcut?
- Thesis pipeline vs. fixed LSTM across frame rates: does the original fps trend survive without the padding bug?
- Worst to best fold, compared with the spread of the original 18-video results (50% to 94%).

Original thesis test accuracy (18 videos each; * = the two runs that used the wrong frames or checkpoint):

| | 5 fps | 15 fps | 30 fps |
|---|---|---|---|
| Model A, 1000 frames | 50% | 78% | 83% |
| Model A, 500 frames | 56% | 89%* | 94% |
| Model B, 1000 frames | 83% | 78% | 72% |
| Model B, 500 frames | 78% | 67%* | 61% |

## 10. Pre-theft test

Annotated shoplifting videos are cut `PRE_THEFT_MARGIN_SEC` before the theft starts, so only the lead-up is seen. Normal videos are cut to lengths drawn from the same distribution, so the length baseline should drop to chance. Models are trained on whole videos and only tested on the cut clips.

- Detection rate: share of pre-theft clips scored above 0.5.
- False alarm rate: share of cut normal clips scored above 0.5.

If the one-frame baseline does as well here as the LSTM, the models are recognising the scene, not anticipating the theft. Only about 20 videos qualify, so the uncertainty is large.

In [ ]:
if n_pos == 0:
    print("No videos qualify for the pre-theft test in this run (expected with SMOKE_TEST = True).")
else:
    d = ALL[(ALL.view == "pre_theft") & ALL.video.isin(list(PRE_CUT))]
    rows = []
    for name, g in d.groupby("experiment", sort=False):
        per_seed = [(float((s.prob[s.label == 1] > 0.5).mean()), float((s.prob[s.label == 0] > 0.5).mean()), auc(s))
                    for _, s in g.groupby("seed")]
        detect, false_alarm, a = np.nanmean(per_seed, axis=0)
        rows.append({"experiment": name, "detection rate": detect, "false alarm rate": false_alarm, "AUC": a})
    display(pd.DataFrame(rows).set_index("experiment").round(2))
    plot_results(ALL, view="pre_theft")